# 09_seeds — H5만 seed 17·73 추가 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 4.3절: seed 반복은 판정 경계(±.01) 근처 결과에만 한다. 08에서 경계에 걸린 것은 **H5**(R1-집단 − R1-집단-A, 외부 test +.009 대 기준 +.01) 하나다. kNN은 무작위성이 없어 R1-집단·R1-집단-A만 다시 학습한다.

- **학습:** 03b·03c와 같은 데이터·설정·학습 함수, seed만 17 또는 73. 결과는 Drive `ours/models/seed<N>/<모델>/`.
- **점수:** 07a와 같은 방식으로 dev·test·RouterArena 점수 → `scores-modernbert-seeds/seed<N>/`.
- **평가(CPU):** 각 seed 모델을 07b와 똑같이 dev에서 보정하고 λ를 맞춘 뒤(설정은 다시 고르지 않음), 08과 같은 bootstrap 표본으로 test를 평가해 **세 seed 평균**으로 H5를 다시 판정한다 → `seeds/`.

**런타임: T4 GPU.** 한 seed(모델 2개)에 약 3~4시간. 세션 두 개를 열어 하나는 `SEED = 17`, 하나는 `SEED = 73`으로 동시에 돌리면 된다. 시간이 부족하면 `RUNS`를 하나씩 나눠도 된다. 평가(4절)는 두 seed가 모두 끝난 뒤 한 번만, CPU로도 된다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-seeds.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-train-colab.txt')], check=True)

## 1. 이 세션의 seed

설치는 새 프로세스에서 바로 쓰이므로 세션을 다시 시작하지 않아도 된다.

In [ ]:
SEED = 17                         # 다른 세션에서는 73
RUNS = ['r1_group', 'r1_group_a']  # 나눠 돌릴 때는 하나만

from pathlib import Path
import json, os, shutil, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
assert SEED in (17, 73)
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'), PYTHONUNBUFFERED='1')

def run_logged(script, args):
    """Run a script and stream its log into this cell."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final' / script), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

## 2. 학습 (모델당 1.5~2시간)

모델마다 끝나면 Drive로 복사한다. 이미 Drive에 있는 모델은 건너뛴다(연결이 끊겨 다시 실행할 때).

In [ ]:
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
for run in RUNS:
    final = RUN / 'ours' / 'models' / f'seed{SEED}' / run
    if final.exists():
        print('이미 있음:', final); continue
    local = Path(f'/content/seed{SEED}-{run}')
    shutil.rmtree(local, ignore_errors=True)
    run_logged('ours.py', ['train', '--root', REPO, '--data-dir', RUN / 'ours' / 'data', '--run', run,
                           '--output-dir', local, '--device', 'cuda', '--seed', SEED])
    final.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local, final)
    print('저장:', final, sorted(p.name for p in final.iterdir()))

## 3. 점수 (모델당 10분 안팎)

학습이 끝난 모델만 점수를 낸다. 이미 끝난 점수는 건너뛴다.

In [ ]:
run_logged('score_modernbert.py', ['--run-dir', RUN, '--output-dir', RUN / 'scores-modernbert-seeds' / f'seed{SEED}',
                                   '--seed', SEED, '--device', 'cuda'])
print(json.loads((RUN / 'scores-modernbert-seeds' / f'seed{SEED}' / 'manifest.json').read_text())['outputs'])

## 4. 평가 (두 seed가 모두 끝난 뒤 한 번, CPU로도 된다)

먼저 dev 리허설(test를 읽지 않음, Colab 디스크에만 씀), 이어서 test. `seeds/`가 이미 있으면 멈춘다.

In [ ]:
DATA_ROOT = RUN / 'inputs' / 'test-data'
rehearsal = Path('/content/rehearse-09')
shutil.rmtree(rehearsal, ignore_errors=True)
run_logged('seeds.py', ['--run-dir', RUN, '--repo', REPO, '--data-root', DATA_ROOT, '--output-dir', rehearsal,
                        '--rehearse-on-dev', '--iterations', 500])
print((rehearsal / 'seeds-summary.md').read_text())

In [ ]:
FINAL = RUN / 'seeds'
assert not FINAL.exists(), '이미 평가했다. 다시 하려면 폴더를 다른 이름으로 옮긴 뒤 실행'
run_logged('seeds.py', ['--run-dir', RUN, '--repo', REPO, '--data-root', RUN / 'inputs' / 'test-data', '--output-dir', FINAL])
print((FINAL / 'seeds-summary.md').read_text())